In [1]:
# mount google drive
from google.colab import drive 
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# import cell for all the imports i am using during the execuiton
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from itertools import combinations
import os
import sys
import os
import torch 
import re
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms
import pandas as pd
from collections import defaultdict
from pathlib import Path
import csv

In [3]:
# Constants block

# paths and names 
DATASET_NAME = "filtered_dataset_ferret_3"
MORPHING_MODEL = "mor_diff" # either "face_morpher" or "mor_diff"
ALIGNED_DATASET_PATH = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_NAME}"
MORPHS_PATH = f"/content/drive/MyDrive/Morphs/{MORPHING_MODEL}/{DATASET_NAME}"
results_csv_path = '/content/drive/MyDrive/Morph_Evaluation/results_summary.csv'

# FR model to use (setting as environmental variable cause it's used by python block later)
os.environ["FR_MODEL_NAME"] = "ElasticFaceArc"

# masking parameters
MASKING_METHOD = "none" # either "sharp" or "blur" or "none" for the control group
BAD_PROBES = () # here probes used for morphing would be added, as they shal not be used anywhere else to avoid data leakage
TARGET_FMR = 0.01 # target false match rate for evaluation. As per EU standards it can be either a 0.1% or 1% FMR, so either 0.001 or 0.01
masking_range = (5,8) # provide a range from 0 to 12 for now step is 3
blur_kernel_size = 5 # provide an odd integer for the kernel size

# if method is "sharp" then the following masking parameters should be provided 
if MASKING_METHOD == "sharp" and not masking_range:
    raise ValueError("If masking method is 'sharp', you must provide a masking range.")

# if method is blur then the following masking parameters should be provided
if MASKING_METHOD == "blur" and (not masking_range or blur_kernel_size is None):
    raise ValueError("If masking method is 'blur', you must provide a masking range and a blur kernel size.")

In [4]:
%%bash
# Clonning elasticface and getting the pre-trained model checkpoint
# this one is bash script with strict exit on error

set -e

# clonning elasticface repo if it's not cloned already
if [ ! -d "ElasticFace" ]; then
    git clone "https://github.com/fdbtrs/ElasticFace"
fi

# installing gdown to get stuff from google drive
pip install -q gdown
# downloading the pre-trained model and doing some cleanup cause unfrotunately google drive just works that way
mkdir -p checkpoints

if [[ $FR_MODEL_NAME = "ElasticFaceArc" ]]; then
    gdown 'https://drive.google.com/drive/folders/1q3ws_BQLmgXyiy2msvHummXq4pRqc1rx' --folder -O checkpoints
else
    gdown 'https://drive.google.com/drive/folders/1ZiLLZXQ1jMzFwMGhYjtMwcdHmuedQb-2' --folder -O checkpoints
fi

ls
# Clean up and rename
rm -f checkpoints/training.log
mv checkpoints/*.pth checkpoints/ElasticFaceArc.pth 2>/dev/null || true

echo "Setup completed successfully"

Processing file 1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb 295672backbone.pth
Processing file 1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH training.log
checkpoints
drive
ElasticFace
sample_data
Setup completed successfully


Cloning into 'ElasticFace'...
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb
From (redirected): https://drive.google.com/uc?id=1n0o1SGhtr_X7Y3scI6VFIUa6KW8aAQtb&confirm=t&uuid=faf7c7a7-377f-411a-afb6-007612faa328
To: /content/checkpoints/295672backbone.pth
100%|██████████| 261M/261M [00:02<00:00, 110MB/s]  
Downloading...
From: https://drive.google.com/uc?id=1jGm6rHh-jJ40c34u5eXBgAhR3u4KHblH
To: /content/checkpoints/training.log
100%|██████████| 788k/788k [00:00<00:00, 154MB/s]
Download completed


In [5]:
# -------------------------------------------------------------
# 1. Corrected Masking Function (Channel-wise DFT as in Section 3.2)
# -------------------------------------------------------------
def create_sharp_mask(image_path: str, masking_range: tuple[int, int], target_size=(112, 112)):
    # Load image and resize to model's expected 112x112 input
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Ensure RGB order
    
    h, w, _ = img.shape
    crow, ccol = h // 2, w // 2
    
    # Compute L2 radial distance matrix from center
    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - ccol)**2 + (y - crow)**2)
    
    # 2D Mask: 0 inside [r_min, r_max], 1 elsewhere
    mask_2d = np.ones((h, w), dtype=np.float32)
    mask_2d[(radius >= masking_range[0]) & (radius <= masking_range[1])] = 0.0
    
    # Channel-wise DFT (OpenCV expects 1-channel for 2D real DFT)
    channels = cv2.split(np.float32(img))
    masked_channels = []
    for ch in channels:
        dft = cv2.dft(ch, flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)
        dft_masked = dft_shift * mask_2d[:, :, np.newaxis] # Zero target band
        dft_ishift = np.fft.ifftshift(dft_masked)
        rec_ch = cv2.idft(dft_ishift, flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
        masked_channels.append(rec_ch)
        
    reconstructed_img = cv2.merge(masked_channels)
    
    # Convert to PyTorch Tensor and normalize to [-1, 1] (ElasticFace standard)
    tensor_img = torch.from_numpy(reconstructed_img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0) # Shape: (1, 3, 112, 112)

def preprocess_unmasked_image(image_path: str, target_size=(112, 112)):
    """Preprocess the unmasked trusted probe image identically."""
    img = cv2.imread(image_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor_img = torch.from_numpy(img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0) # Shape: (1, 3, 112, 112) ALREADY TENSOR

In [6]:
def create_blur_mask(
    image_path: str,
    masking_range: tuple[int, int],
    blur_kernel_size: int,
    target_size=(112, 112)
):
    """Mask a radial DFT band and blur the reconstructed image.

    The frequency-band removal matches create_sharp_mask. The resulting
    spatial image is then blurred with the requested Gaussian kernel before
    being converted to the normalized ElasticFace input tensor.
    """
    if len(masking_range) != 2 or masking_range[0] < 0 or masking_range[1] < masking_range[0]:
        raise ValueError("masking_range must be a non-negative (min_radius, max_radius) tuple.")

    if blur_kernel_size <= 0 or blur_kernel_size % 2 == 0:
        raise ValueError("blur_kernel_size must be a positive odd integer.")

    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    crow, ccol = h // 2, w // 2

    # Build the same radial frequency mask used by create_sharp_mask.
    y, x = np.ogrid[:h, :w]
    radius = np.sqrt((x - ccol) ** 2 + (y - crow) ** 2)
    mask_2d = np.ones((h, w), dtype=np.float32)
    mask_2d[(radius >= masking_range[0]) & (radius <= masking_range[1])] = 0.0

    # Apply the frequency mask independently to each color channel.
    channels = cv2.split(np.float32(img))
    reconstructed_channels = []
    for channel in channels:
        dft = cv2.dft(channel, flags=cv2.DFT_COMPLEX_OUTPUT)
        dft_shift = np.fft.fftshift(dft)
        dft_masked = dft_shift * mask_2d[:, :, np.newaxis]
        dft_ishift = np.fft.ifftshift(dft_masked)
        reconstructed_channel = cv2.idft(
            dft_ishift,
            flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT
        )
        reconstructed_channels.append(reconstructed_channel)

    reconstructed_img = cv2.merge(reconstructed_channels)
    reconstructed_img = np.clip(reconstructed_img, 0, 255).astype(np.uint8)

    # GaussianBlur requires an odd positive kernel in both dimensions.
    blurred_img = cv2.GaussianBlur(
        reconstructed_img,
        (blur_kernel_size, blur_kernel_size),
        sigmaX=0
    )

    tensor_img = torch.from_numpy(blurred_img).permute(2, 0, 1).contiguous().float()
    tensor_img = (tensor_img / 255.0 - 0.5) / 0.5
    return tensor_img.unsqueeze(0)  # Shape: (1, 3, 112, 112)


In [7]:
# Loading the model 

# appending repo to the path to be able to import code from it
sys.path.append("./ElasticFace")
print(os.listdir("."))
from backbones.iresnet import iresnet100 # imporing the iresnet 100 architecture from the repo


# selecting the device to run the model on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# initializing the model and moving it to the selected device
model = iresnet100()
model.to(device)

# loading the checkpoint into the model architecture
checkpoint = torch.load("./checkpoints/ElasticFaceArc.pth", map_location=device)
model.load_state_dict(checkpoint)

# switching the model to evaluation mode
model.eval()

# testing if the model is working by passing a fake tensor through it
fake_input = torch.randn(1, 3, 112, 112).to(device)
with torch.no_grad():
    fake_output = model(fake_input)
print(f"Fake output shape: {fake_output.shape}, if that works, the model is loaded and working correctly.")

['.config', 'ElasticFace', 'checkpoints', 'drive', 'sample_data']
Using device: cuda
Fake output shape: torch.Size([1, 512]), if that works, the model is loaded and working correctly.


In [8]:
# get ALL images from the aligned dataset as a list of paths by walking its structure
image_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(ALIGNED_DATASET_PATH)
    for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

print(image_files[:5])  # print the first 5 image paths to verify

# Returns a list of dicts, one per morph:
# [{"morph": morph_path, "verification1": image_path, "verification2": image_path}, ...]
# The exact source probe names are extracted from the morph filename and excluded,
# while the remaining same-identity images are used as verification probes.
def get_evaluation_probes(morphs_path: str = MORPHS_PATH) -> list[dict]:
    results = []
    global BAD_PROBES  # Ensure we modify the global BAD_PROBES variable

    for morph in sorted(os.listdir(morphs_path)):
        if not morph.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        # getting the full path to the morph and it's name without the extension
        morph_path = os.path.join(morphs_path, morph)
        morph_name = os.path.splitext(os.path.basename(morph))[0]

        # using regex, extracting the identities which were used to construct the morph
        identity_1_name = re.search(r"(?<=^morphed_).*(?=_and)", morph_name).group(0)
        identity_2_name = re.search(r"(?<=_and_).*(?=$)", morph_name).group(0)

        # adding those identities to the BAD_PROBES tuple to avoid data leakage
        BAD_PROBES += (identity_1_name, identity_2_name)

        # getting identity prefixes to find other probes of the same identity
        identity_1_prefix = identity_1_name.split("_")[0]
        identity_2_prefix = identity_2_name.split("_")[0]

        # getting the paths to the other probes of the identities, excluding the morph identity itself
        identity_1_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_1_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_1_name
        )
        identity_2_probes = sorted(
            file for file in image_files
            if os.path.basename(file).startswith(f"{identity_2_prefix}_")
            and os.path.splitext(os.path.basename(file))[0] != identity_2_name
        )

        if not identity_1_probes or not identity_2_probes:
            continue

        # getting first probe of each identity to use for verification
        results.append({
            "morph": morph_path,
            "verification1": identity_1_probes[0],
            "verification2": identity_2_probes[0],
            "fnmr_stash": [identity_1_probes[1], identity_2_probes[1]] # since we are having 3 photos per identity, stashing the third already not to find them again after
        })

    return results


['/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fb.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940928_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00005/00005_930831_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00005/00005_941121_fa.png']


In [9]:
# getting the evaluation probes for the morphs in the dataset
evaluation_probes = get_evaluation_probes(MORPHS_PATH)
print(evaluation_probes[0])
print(BAD_PROBES[0])

{'morph': '/content/drive/MyDrive/Morphs/mor_diff/filtered_dataset_ferret_3/morphed_00002_940128_fa_and_00184_940422_fa.png', 'verification1': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fb.png', 'verification2': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00184/00184_940128_fa.png', 'fnmr_stash': ['/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940928_fa.png', '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00184/00184_940128_fb.png']}
00002_940128_fa


In [10]:
# Functions for batch processing images for FMR calibration and FNMR calculation later.
def build_tensor(image_path: str, masked: bool):
    if not masked:
        return preprocess_unmasked_image(image_path)

    match MASKING_METHOD:
        case "none":
            # Baseline: unmasked reference photo.
            return preprocess_unmasked_image(image_path)

        case "sharp":
            # MR-FR: mask the reference photo.
            return create_sharp_mask(image_path, masking_range)

        case "blur":
            # MR-FR: mask and blur the reference photo.
            return create_blur_mask(
                image_path, masking_range, blur_kernel_size
            )

        case _:
            raise ValueError(f"Unknown masking method: {MASKING_METHOD}")


def compute_embeddings(paths, masked, batch_size=32):
    """Compute normalized embeddings while processing at most 32 images per batch."""
    embeddings = []

    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        batch = torch.cat([
            build_tensor(path, masked)
            for path in batch_paths
        ]).to(device) # moving to gpu here

        with torch.inference_mode():
            output = model(batch)
            output = F.normalize(output, p=2, dim=1)

        embeddings.append(output)

    if not embeddings:
        return torch.empty((0, 0), device=device)

    return torch.cat(embeddings)

In [11]:
# Perform calibration for the requested FMR value and save the threshold.
# The threshold is reused for the later FNMR match/not-match decisions.

# Build a dictionary containing one calibration image per identity.
calibration_dict = {}

for image_path in image_files:
    identity_name = image_path.split("/")[-2]
    if identity_name in calibration_dict:
        continue

    # Exclude identities used to create morphs to avoid data leakage.
    if identity_name in BAD_PROBES:
        continue

    calibration_dict[identity_name] = image_path

print(f"Calibration dict: {calibration_dict}")

CALIBRATION_THRESHOLD = None
model.eval()

# compute_embeddings expects image paths, while calibration_dict keys are identities.
calibration_paths = list(calibration_dict.values())
print(f"Calibrating on {len(calibration_paths)} identities...")

# The first image is processed according to MASKING_METHOD.
masked_embeddings = compute_embeddings(calibration_paths, masked=True)

# The second image is always an unmasked live probe.
unmasked_embeddings = compute_embeddings(calibration_paths, masked=False)

# Every row/column pair gives one impostor score.
score_matrix = masked_embeddings @ unmasked_embeddings.T

# Keep only different-identity comparisons.
upper_triangle = torch.triu(
    torch.ones_like(score_matrix, dtype=torch.bool),
    diagonal=1,
)

impostor_scores = score_matrix[upper_triangle].detach().cpu().numpy()
impostor_scores = np.asarray(impostor_scores)

print(f"Total impostor comparisons computed: {len(impostor_scores)}")

percentile = (1.0 - TARGET_FMR) * 100.0
CALIBRATION_THRESHOLD = float(np.percentile(impostor_scores, percentile))
print(
    f"Calibration completed. Threshold for FMR={TARGET_FMR}: "
    f"{CALIBRATION_THRESHOLD:.4f}"
)

Calibration dict: {'00002': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00002/00002_940128_fa.png', '00005': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00005/00005_930831_fa.png', '00013': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00013/00013_930831_fb.png', '00019': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00019/00019_940128_fa.png', '00020': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00020/00020_930831_fa.png', '00024': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00024/00024_930831_fa.png', '00028': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00028/00028_930831_fb.png', '00029': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00029/00029_940128_fa.png', '00038': '/content/drive/MyDrive/Datasets_aligned/filtered_dataset_ferret_3/00038/00038_930831_fb.png', '00040': '/content/drive/MyDrive/Datasets_ali

In [12]:
def get_identity(image_path: str) -> str:
    return Path(image_path).parent.name


# Group every dataset image by identity.
images_by_identity = defaultdict(list)

# get dictionary of all images per identity where key is identity name 
for image_path in image_files:
    identity = get_identity(image_path)
    images_by_identity[identity].append(image_path)


# Preserve the probes selected for morph evaluation.
live_by_identity = defaultdict(set)
legit_by_identity = defaultdict(set)

for morphtriplet in evaluation_probes:
    # getting verification paths for the morph
    verification_paths = [
        morphtriplet[f"verification{i}"]
        for i in range(1, 3)
    ]

    # getting "key for the image"
    for image_path in verification_paths:
        identity = get_identity(image_path)
        live_by_identity[identity].add(image_path) # add them to respective live identities

    # do the same for the stashed things we haven't used anywhere
    for image_path in morphtriplet["fnmr_stash"]:
        identity = get_identity(image_path)
        legit_by_identity[identity].add(image_path)


# Add identities that were never used in a morph.
# For these identities, use one image as a live probe and one as a legit probe.
# morph identities is a set union -- so all identity names without duplicates
morph_identities = set(live_by_identity) | set(legit_by_identity)

for identity, paths in images_by_identity.items():
    if identity in morph_identities:
        continue # if identity is already one of the morphed, it's probes are already added and we don't need it second time

    # if that's not the case, just add first 2 probes for this identity
    paths = sorted(paths)

    if len(paths) >= 2:
        live_by_identity[identity].add(paths[0])
        legit_by_identity[identity].add(paths[1])

print("Live probes:", len(live_by_identity))
print("Legit probes:", len(legit_by_identity))

Live probes: 107
Legit probes: 107


In [13]:
# Calculate FNMR using identity-preserving legit/live probe pairs.
# Each legit image is compared only with live images from the same identity.
if CALIBRATION_THRESHOLD is None:
    raise RuntimeError("CALIBRATION_THRESHOLD must be calculated before FNMR evaluation.")

# Keep only identities that have both kinds of probe available.
matching_identities = sorted(
    set(live_by_identity) & set(legit_by_identity)
)

# Build every genuine comparison for each identity. Sets remove duplicate paths;
# sorting makes the evaluation order deterministic.
comparison_pairs = [
    (identity, legit_path, live_path)
    for identity in matching_identities
    for legit_path in sorted(legit_by_identity[identity])
    for live_path in sorted(live_by_identity[identity])
]

if not comparison_pairs:
    raise RuntimeError("No same-identity legit/live pairs are available for FNMR.")

# Compute each unique image embedding once, then reuse it for all pairs.
legit_paths = sorted({legit_path for _, legit_path, _ in comparison_pairs})
live_paths = sorted({live_path for _, _, live_path in comparison_pairs})

# Legit passport probes are processed with the selected masking method.
legit_embeddings = compute_embeddings(
    legit_paths,
    masked=True,
    batch_size=32,
)

# Live capture probes are always processed without masking.
live_embeddings = compute_embeddings(
    live_paths,
    masked=False,
    batch_size=32,
)

legit_embedding_by_path = {
    path: embedding
    for path, embedding in zip(legit_paths, legit_embeddings)
}
live_embedding_by_path = {
    path: embedding
    for path, embedding in zip(live_paths, live_embeddings)
}

# Normalized embeddings make the dot product equal to cosine similarity.
fnmr_rows = []
for identity, legit_path, live_path in comparison_pairs:
    similarity = float(
        torch.dot(
            legit_embedding_by_path[legit_path],
            live_embedding_by_path[live_path],
        )
    )
    is_match = similarity >= CALIBRATION_THRESHOLD

    fnmr_rows.append({
        "identity": identity,
        "legit_path": legit_path,
        "live_path": live_path,
        "similarity": similarity,
        "match": is_match,
        "decision": "match" if is_match else "not_match",
    })

fnmr_results_df = pd.DataFrame(fnmr_rows)
false_non_matches = (~fnmr_results_df["match"]).sum()
total_genuine_comparisons = len(fnmr_results_df)
FNMR = false_non_matches / total_genuine_comparisons

print(f"Identities evaluated: {len(matching_identities)}")
print(f"Genuine comparisons: {total_genuine_comparisons}")
print(f"False non-matches: {false_non_matches}")
print(f"FNMR: {FNMR:.6f} ({FNMR:.2%})")
print(fnmr_results_df.head())

Identities evaluated: 107
Genuine comparisons: 107
False non-matches: 0
FNMR: 0.000000 (0.00%)
  identity                                         legit_path  \
0    00002  /content/drive/MyDrive/Datasets_aligned/filter...   
1    00005  /content/drive/MyDrive/Datasets_aligned/filter...   
2    00013  /content/drive/MyDrive/Datasets_aligned/filter...   
3    00019  /content/drive/MyDrive/Datasets_aligned/filter...   
4    00020  /content/drive/MyDrive/Datasets_aligned/filter...   

                                           live_path  similarity  match  \
0  /content/drive/MyDrive/Datasets_aligned/filter...    0.833434   True   
1  /content/drive/MyDrive/Datasets_aligned/filter...    0.902348   True   
2  /content/drive/MyDrive/Datasets_aligned/filter...    0.853921   True   
3  /content/drive/MyDrive/Datasets_aligned/filter...    0.827461   True   
4  /content/drive/MyDrive/Datasets_aligned/filter...    0.732223   True   

  decision  
0    match  
1    match  
2    match  
3    match 

In [14]:
# Image tensor transformation pipeline to prepare the images for the model
transform_pipeline = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(), # going from [0, 255] to [0.0, 1.0]
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalizing to [-1, 1]
])

In [15]:
# 2 funcitons which are used to determine how similar are the morphs to original people 
# first one gets the embedding of the image 
# second calculates the cosine similarity between two embeddings
def get_embedding(image_path: str) -> torch.Tensor:
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform_pipeline(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(image_tensor)

    if isinstance(embedding, (tuple, list)):
        embedding = embedding[0]

    # keep it on gpu
    embedding = embedding.detach().float()

    if embedding.ndim > 1:
        embedding = embedding.reshape(embedding.shape[0], -1)
        if embedding.shape[0] == 1:
            embedding = embedding[0]

    return embedding


def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    a = a.view(-1)
    b = b.view(-1)
    denom = (torch.norm(a) * torch.norm(b)).clamp_min(1e-12)
    return float(torch.dot(a, b) / denom)

In [16]:
# make sure the model is in eval mode
model.eval()

match MASKING_METHOD: 
    case "none":
        CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity_{MASKING_METHOD}.csv"
    case "sharp": 
        CSV_OUTPUT_PATH = f"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity_{MASKING_METHOD}_{masking_range[0]}_{masking_range[1]}.csv"
    case "blur": 
        CSV_OUTPUT_PATH = F"/content/drive/MyDrive/Morph_Evaluation/{DATASET_NAME}_cosine_similarity_{MASKING_METHOD}_{masking_range[0]}_{masking_range[1]}_{blur_kernel_size}.csv"


os.makedirs(os.path.dirname(CSV_OUTPUT_PATH), exist_ok=True)


rows = []
fooled_counter = 0
for item in evaluation_probes:
    morph_path = item["morph"]
    verification1_path = item["verification1"]
    verification2_path = item["verification2"]

    if morph_path is None or verification1_path is None or verification2_path is None:
        continue

    # siwtch on how are we going to process the morph based on the selected masking method 
    match MASKING_METHOD:
        case "none":
            morph_embedding = get_embedding(morph_path)
        case "sharp":
            morph_masked_tensor = create_sharp_mask(morph_path, masking_range).to(device)
            with torch.no_grad():
                morph_embedding = model(morph_masked_tensor)
        case "blur":
                morph_masked_tensor = create_blur_mask(
                    morph_path, masking_range, blur_kernel_size
                ).to(device)
                with torch.no_grad():
                    morph_embedding = model(morph_masked_tensor)
        case _:
            raise ValueError(f"Unknown masking method: {MASKING_METHOD}")
        
    verification1_embedding = get_embedding(verification1_path)
    verification2_embedding = get_embedding(verification2_path)

    sim1 = cosine_similarity(morph_embedding, verification1_embedding)
    sim2 = cosine_similarity(morph_embedding, verification2_embedding)

    fooled = False 

    if sim1 >= CALIBRATION_THRESHOLD and sim2 >= CALIBRATION_THRESHOLD:
        fooled = True
        fooled_counter += 1 

    rows.append({
        "morph": morph_path,
        "verification1": verification1_path,
        "verification2": verification2_path,
        "1_similarity": sim1,
        "2_similarity": sim2,
        "fooled": fooled
    })

results_df = pd.DataFrame(rows, columns=["morph", "verification1", "verification2", "1_similarity", "2_similarity", "fooled"])
results_df.to_csv(CSV_OUTPUT_PATH, index=False)

print(f"Saved {len(results_df)} rows to: {CSV_OUTPUT_PATH}")
print(results_df.head())


# calculating MMPMR (Morph Match Presentation Match Rate) for the dataset
MMPMR = round(results_df["fooled"].value_counts().get(True)/len(results_df),2)
print(f"MMPMR for this dataset is: {MMPMR}")

Saved 46 rows to: /content/drive/MyDrive/Morph_Evaluation/filtered_dataset_ferret_3_cosine_similarity_none.csv
                                               morph  \
0  /content/drive/MyDrive/Morphs/mor_diff/filtere...   
1  /content/drive/MyDrive/Morphs/mor_diff/filtere...   
2  /content/drive/MyDrive/Morphs/mor_diff/filtere...   
3  /content/drive/MyDrive/Morphs/mor_diff/filtere...   
4  /content/drive/MyDrive/Morphs/mor_diff/filtere...   

                                       verification1  \
0  /content/drive/MyDrive/Datasets_aligned/filter...   
1  /content/drive/MyDrive/Datasets_aligned/filter...   
2  /content/drive/MyDrive/Datasets_aligned/filter...   
3  /content/drive/MyDrive/Datasets_aligned/filter...   
4  /content/drive/MyDrive/Datasets_aligned/filter...   

                                       verification2  1_similarity  \
0  /content/drive/MyDrive/Datasets_aligned/filter...      0.612927   
1  /content/drive/MyDrive/Datasets_aligned/filter...      0.550737   
2  /c

In [17]:
# create a csv file if not exists and put the header in it
# then just add results to it.

row = [
    DATASET_NAME,
    MORPHING_MODEL,
    MASKING_METHOD,
    masking_range[0] if masking_range else "",
    masking_range[1] if masking_range else "",
    blur_kernel_size if MASKING_METHOD == "blur" else "",
    TARGET_FMR,
    FNMR,
    MMPMR,
]

results_path = Path(results_csv_path)
file_exists = results_path.exists()

with results_path.open("a+", newline="") as f:
    f.seek(0)
    existing_rows = list(csv.reader(f))

    # Compare only the experiment configuration, excluding result metrics.
    experiment_key = tuple(row[:6])
    already_exists = any(tuple(existing[:6]) == experiment_key for existing in existing_rows)

    if not already_exists:
        f.seek(0, 2)  # Move to the end before writing
        writer = csv.writer(f)

        if not file_exists or not existing_rows:
            writer.writerow([
                "dataset",
                "morphing_model",
                "masking_method",
                "masking_range_start",
                "masking_range_end",
                "blur_kernel_size",
                "target_fmr",
                "fnmr",
                "mmpmr",
            ])

        writer.writerow(row)
        print("New experiment written.")
    else:
        print("Experiment already exists; skipping.")

New experiment written.
